> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 6 · Notebook 05 — Second-order Greeks and finite differences

**Sessions:** S5 (Second-order Greeks) · [Lesson plan](../../docs/lessons/PART_06_FUTURES_OPTIONS_ENGINEERING.md) · graded labs in [`labs/part06/`](../../labs/part06/)

**You will:**
1. Compute gamma by bump-and-revalue, and pick the bump size.
2. Write vanna and volga in closed form, and verify them by finite differences.
3. Watch gamma, theta and charm explode as expiry approaches.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with known parameters, so every estimate can be compared with the truth. Units: T in years, σ as a decimal, vega per 1.00 σ, theta per year, `cp = +1` call / `−1` put.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p6lib.py is in notebooks/part06/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p6lib as p

p.use_course_style()

In [ ]:
base = dict(S=600.0, K=600.0, T=30 / 365, r=0.045, q=0.013, sigma=0.18, cp=1)
price = p.bsm_price

## 1. Gamma by bump-and-revalue

Any Greek of any pricer can be estimated by re-pricing with bumped inputs. For gamma, the central second difference: `Γ ≈ (V(S + h) − 2V(S) + V(S − h)) / h²`. `args` is a dict of the pricer's keyword arguments.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def fd_gamma(pricer, args, h):
    up, dn = dict(args, S=args["S"] + h), dict(args, S=args["S"] - h)
    return (pricer(**up) - 2 * pricer(**args) + pricer(**dn)) / h ** 2

hs = [5.0, 1.0, 0.1]
mine = [fd_gamma(price, base, h) for h in hs]
mine = p.check("fd_gamma", mine, [p.bump(price, base, "S", h, order=2) for h in hs])
exact = p.greeks(**base)["gamma"]
pd.DataFrame({"h": hs, "finite difference": mine, "error vs closed form": np.array(mine) - exact})

Smaller isn't always better. The truncation error shrinks like `h²`, but the rounding error of subtracting nearly equal prices grows like `ε/h²`. Sweep `h` and the error traces a **U**:

In [ ]:
hh = np.logspace(-5, 1.5, 60)
err = [abs(p.bump(price, base, "S", h, order=2) - exact) for h in hh]
fig, ax = plt.subplots()
ax.loglog(hh, err)
ax.set(xlabel="bump h (price units)", ylabel="|FD gamma − exact|", title="Truncation error on the right, rounding error on the left"); plt.show()
h_rel = 1e-3 * base["S"]
print(f"best h here ≈ {hh[int(np.argmin(err))]:.3g}; a bump of 0.1% of spot (h = {h_rel:.1f}) gives an error of "
      f"{abs(p.bump(price, base, 'S', h_rel, order=2) - exact):.1e}, with gamma itself {exact:.1e}")

## 2. Vanna and volga

* **vanna** = ∂Δ/∂σ = ∂vega/∂S = `−e^{−qT} n(d1) d2 / σ`: how delta moves when vol moves (skew risk)
* **volga** (vomma) = ∂vega/∂σ = `vega · d1 · d2 / σ`: the convexity of the price in vol (wing risk)

`vega` is `p.greeks(...)["vega"]` (raw, per 1.00 σ).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def vanna_volga(S, K, T, r, q, sigma, cp):
    d1, d2 = p.d1d2(S, K, T, r, q, sigma)
    vega = p.greeks(S, K, T, r, q, sigma, cp)["vega"]
    vanna = -np.exp(-q * T) * p.n(d1) * d2 / sigma
    volga = vega * d1 * d2 / sigma
    return vanna, volga

Ks = np.array([540.0, 600.0, 660.0])
args = dict(base, K=Ks)
mine = list(vanna_volga(**args))
ref = p.second_order(**args)
mine = p.check("vanna and volga", mine, [ref["vanna"], ref["volga"]])
fd = [p.bump(lambda **a: p.greeks(**a)["delta"], args, "sigma", 1e-4), p.bump(lambda **a: p.greeks(**a)["vega"], args, "sigma", 1e-4)]
pd.DataFrame({"K": Ks, "vanna": mine[0], "vanna (FD)": fd[0], "volga": mine[1], "volga (FD)": fd[1]}).round(4)

Vanna has opposite signs in the two wings: a downside put's delta moves one way when vol rises, an upside call's the other. Volga is near zero at the money and largest in the wings, which is why wing options carry a vol-of-vol premium.

## 3. The last week

An ATM option's gamma and theta both grow like `1/√T`. **Charm** (how delta drifts as time passes, per year) makes a delta-hedged position drift over a weekend even if the stock doesn't move.

In [ ]:
dte = np.arange(60, 0, -1)
rows = []
for d in dte:
    g = p.to_display(p.greeks(600.0, 600.0, d / 365, 0.045, 0.013, 0.18, 1))
    g2 = p.second_order(605.0, 600.0, d / 365, 0.045, 0.013, 0.18, 1)
    rows.append({"DTE": d, "ATM gamma": g["gamma"], "ATM theta/day": g["theta"], "charm/day (K=600, S=605)": g2["charm"] / 365})
t = pd.DataFrame(rows).set_index("DTE")
t.plot(subplots=True, figsize=(10, 6), sharex=True, title="Second-order effects near expiry"); plt.gca().invert_xaxis(); plt.show()
print(t.loc[[30, 7, 1]].round(4))

## Wrap-up

* Closed forms where you have them; bump-and-revalue (with a sensible `h`) for everything else, and to test the closed forms.
* Near expiry the second-order Greeks dominate; hedge more often, or hold less gamma.
* Graded version: `labs/part06/week22_greeks_numerics` (eight second-order Greeks verified by finite differences).